# Main figures

Figures 2–8 and Table 1 of the paper, rendered at the journal's artwork size (180 mm wide, 7–8 pt DejaVu Sans, PDF + PNG). Figure 1, the pipeline schematic, is assembled outside Python from insets drawn in Supplementary Note 1.

In [ ]:
import pandas as pd

from cordex2energy_analysis.config import COMPARISON_SCENARIO, RUN_TAG_HIGH_2050, SCENARIO_LABELS, setup
from cordex2energy_analysis.fields import load_country_output, load_demand_by_month
from cordex2energy_analysis.figures_main import (ABSOLUTE_CHANGE_ROWS, CLASS_LABELS, driver_change_table,
                                                 duration_spectrum_table, europe_event_totals,
                                                 exceedance_ratio_table, figure_mean_state,
                                                 figure_scenarios, figure_storage, figure_sub_daily,
                                                 figure_tail_and_duration, mean_state_quantities,
                                                 regime_change_table, severity_balance_table,
                                                 show_table, threshold_table)
from cordex2energy_analysis.config import DURATION_BINS, SOUTHERN_ZONES
from cordex2energy_analysis.plotting_publication import (europe_change_figure, europe_change_summary,
                                                         europe_change_table, markdown_table,
                                                         publication_rc, save)
from cordex2energy_analysis.runs import (annual_event_counts, event_tables, exceedance_by_hour,
                                         load_run, pooled_exceedance_by_hour,
                                         positive_residual_load_share)
from cordex2energy_analysis.tables import (EUROPE_COLUMNS, europe_change_rows, scenario_table,
                                           severity_by_duration, storage_share_table, zone_change_table)

context = setup("paper_publication")

In [ ]:
run = load_run(context)                      # the paper's run, scored on its CERRA p95
tables = event_tables(run, context)
southern_hours = pooled_exceedance_by_hour(exceedance_by_hour(run, context), SOUTHERN_ZONES)
country_output = load_country_output(context)
demand_by_month = load_demand_by_month(context)
quantities = mean_state_quantities(country_output, demand_by_month, context.period_windows)

## Figures 2 and 3 - The mean state
**In:** per-country generation, demand and residual load of the paper's run, nine chains. **Shows:** the level at +1.5 °C and its change at +2.0 and +3.0 °C, supply (Figure 2) and demand with residual load (Figure 3).

In [ ]:
with publication_rc():
    for stem, rows in (("fig2", quantities[:3]), ("fig3", quantities[3:])):
        figure, _ = figure_mean_state(rows, context)
        save(figure, stem, context.figure_dir)
show_table(f"Numbers behind Figures 2 and 3 - change from {context.reference} to {context.warmest}; per cent, "
           "residual load in TWh yr-1",
           zone_change_table(quantities, context.reference, context.warmest, ABSOLUTE_CHANGE_ROWS))

## Figure 4 and Table 1 - The Europe-wide change, demand against supply
**In:** the same fields summed over the 35 modelled countries. **Shows:** each chain's change from +1.5 to +3.0 °C, annually, in winter and in summer (Table 1 lists the values).

In [ ]:
europe_rows = europe_change_rows(country_output, demand_by_month, context.period_windows)
europe_labels = [label for label, _ in europe_rows]
europe_change = europe_change_table(europe_rows, EUROPE_COLUMNS, context.reference, context.warmest)
with publication_rc():
    save(europe_change_figure(europe_change, europe_labels, EUROPE_COLUMNS, value="twh"), "fig4",
         context.figure_dir)

table_1 = europe_change_summary(europe_change, europe_labels, EUROPE_COLUMNS)
pd.concat(table_1, names=["time scale", "quantity"]).to_csv(
    context.figure_dir / "table1_europe_change.csv", float_format="%.2f")
for column, part in table_1.items():
    print(f"Table 1 - {column}\n{markdown_table(part, index_label='quantity')}\n")

## Figure 5 - The deficit tail and event duration
**In:** zonal residual load of the paper's run and the CERRA p95 threshold. **Shows:** the balancing zones, southern exceedance curves, and events and severity by duration.

In [ ]:
with publication_rc():
    save(figure_tail_and_duration(run, context, tables["storage_events"]), "fig5", context.figure_dir)
show_table("CERRA P95 threshold per zone (GWh per 3h step)", threshold_table(run.threshold))
show_table("Numbers behind Figure 5b - exceedance of the threshold and its ratio",
           exceedance_ratio_table(run, context), digits=2)
show_table("Numbers behind Figure 5c, d - Europe-wide events and severity per year by duration",
           duration_spectrum_table(tables["storage_events"], context), digits=2)

## Figure 6 - The sub-daily regime
**In:** sub-daily events, exceedance by hour of day and the driver anomalies over the deficit window. **Shows:** events per zone, their seasonal change, the southern diurnal cycle and its drivers.

In [ ]:
with publication_rc():
    save(figure_sub_daily(tables, southern_hours, context), "fig6", context.figure_dir)
for duration_class, figure_name in (("sub_daily", "Figure 6a, b"), ("multi_day", "Supplementary Figure S31a, b")):
    show_table(f"Numbers behind {figure_name} - {duration_class} events per year, paired change and its "
               "seasonal split (events yr-1)",
               regime_change_table(tables["events_annual"], tables["events_by_season"], context,
                                   duration_class), digits=2)
show_table("Numbers behind Figure 6d - driver contributions to the deficit-window change (GWh per 3h step)",
           driver_change_table(tables["evening_peak_components"], context))

## Figure 7 - What storage balances
**In:** each event's severity with and without reservoirs and batteries, at +3.0 °C. **Shows:** the share balanced per zone and duration class, and the energy balanced and left by duration.

In [ ]:
severity_by_bin = severity_by_duration(tables["storage_events"])
with publication_rc():
    save(figure_storage(tables["storage_attenuation"], severity_by_bin, context), "fig7",
         context.figure_dir)
for period in (context.reference, context.warmest):
    show_table(f"Numbers behind Figure 7a - share of severity balanced by storage at {period} (%)",
               storage_share_table(tables["storage_attenuation"], period, DURATION_BINS, CLASS_LABELS,
                                   run.zones))
show_table(f"Numbers behind Figure 7b - severity balanced and left per duration at {context.warmest}",
           severity_balance_table(severity_by_bin, context.warmest), digits=2)

## Figure 8 - The HIGH/2050 build-out
**In:** the HIGH/2050 run scored on the paper's (LOW/2050) threshold. **Shows:** sub-daily and multi-day events per zone under both build-outs.

In [ ]:
high_run = load_run(context, RUN_TAG_HIGH_2050, threshold=run.threshold)
events_by_scenario = {SCENARIO_LABELS["LOW"]: tables["events_annual"],
                      SCENARIO_LABELS[COMPARISON_SCENARIO]: annual_event_counts(high_run, context)}
with publication_rc():
    save(figure_scenarios(events_by_scenario, context), "fig8", context.figure_dir)

for duration_class, panel in (("sub_daily", "8a"), ("multi_day", "8b")):
    show_table(f"Numbers behind Figure {panel} - {duration_class} events per year; Δ is the paired change",
               scenario_table(events_by_scenario, "freq_per_year", context.reference, context.warmest,
                              run.zones, len(run.models), duration_class=duration_class))
show_table("Europe-wide events per year (sum over zones, median of chains)",
           europe_event_totals(events_by_scenario, context))
residual_load_share = {label: positive_residual_load_share(source, context, label)
                       for label, source in ((SCENARIO_LABELS["LOW"], run),
                                             (SCENARIO_LABELS[COMPARISON_SCENARIO], high_run))}
show_table("Positive residual load as a share of the zone's demand (%); Δ is the paired change",
           scenario_table(residual_load_share, "share_of_demand_pct", context.reference,
                          context.warmest, run.zones, len(run.models)))